# Q-Blueprint — Complete Consolidated Reproducible Notebook
## Closed-loop calibration of a drifting three-level transmon and reported QAOA impact

This notebook consolidates the source code available in the uploaded project and organizes the model, drift simulation, pulse/propagator checks, calibration-study evidence, policy comparison, QAOA findings, and D6 memo in one place.

**Reproducibility boundary:** the supplied bundle contains the D1 physical model and its tests, reports, figures, and aggregate tables. It does **not** contain the original D2–D6 implementation modules for the complete calibration agent, policies, hidden-state evaluator, and QAOA sweep. Therefore, the notebook reruns the supplied core model and tests, while later-stage results are presented as **reported results**, not falsely claimed as recomputed. The embedded code is the exact supplied core source where included; the missing original implementation is not fabricated or represented as original.


## 1. Environment and reproducibility

Recommended environment: Python 3.10+, NumPy, SciPy, Matplotlib, and pytest. This notebook defines the core modules directly in memory, so it does not require sibling `.py` files. The test cell runs a compact set of physical-model checks in notebook context; a separate cell also documents how to run the original supplied test suite when the project directory is available.


In [ ]:
%pip -q install -r ../requirements.txt


## 2. Embed the supplied core source files

The next cell stores the supplied `config.py`, `drift.py`, `transmon.py`, and `propagator.py` source in this notebook and executes them in dependency order. This keeps the notebook self-contained for the D1 model. The source is embedded from the provided project files without changing its physics or parameter values.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy.special
import scipy.signal
from dataclasses import dataclass
from pathlib import Path
import json, math, time, sys, subprocess, textwrap
from IPython.display import display, Image

CORE_SOURCE = {}


In [ ]:
# Supplied source file: config.py
CORE_SOURCE['config.py'] = '"""All physical constants and design choices in one place.\nInternal units: time in ns, angular frequency in rad/ns. Hz only at I/O."""\nfrom dataclasses import dataclass\nimport numpy as np\n\nTWO_PI = 2.0 * np.pi\n\ndef hz_to_radns(f_hz):\n    """Ordinary frequency [Hz] -> angular frequency [rad/ns]."""\n    return TWO_PI * np.asarray(f_hz) * 1e-9\n\ndef radns_to_hz(w):\n    return np.asarray(w) / TWO_PI * 1e9\n\n@dataclass(frozen=True)\nclass Config:\n    # --- specified by the problem statement ---\n    alpha_hz: float = -300e6        # anharmonicity alpha/2pi\n    t_gate_ns: float = 20.0         # gate duration\n    T1_ns: float = 100e3            # 100 us\n    T2_ns: float = 80e3             # 80 us\n    t_shot_s: float = 500e-6        # repetition time per shot\n    p10: float = 0.01               # P(1|0)\n    p01: float = 0.03               # P(0|1); |2> reads as 1\n    day_s: float = 24 * 3600.0\n    eps_th: float = 1e-3\n    # drift model\n    xf_sigma_hz: float = 150e3\n    xf_tau_s: float = 2 * 3600.0\n    rtn_amp_hz: float = 0.8e6\n    rtn_dwell_s: float = 4 * 3600.0\n    gain_sin_amp: float = 0.02\n    xg_sigma: float = 0.005\n    xg_tau_s: float = 3 * 3600.0\n    official_seed: int = 2026\n    # --- design choices (documented in the report) ---\n    sigma_frac: float = 0.25        # sigma = tg/4 (specified)\n    n_steps: int = 200              # propagator steps per gate (dt = 0.1 ns); convergence-tested\n    drift_dt_s: float = 1.0         # drift sampling grid\n    drift_margin_s: float = 3600.0  # trace extends past 24 h so calibration at end is defined\n\n    @property\n    def alpha(self):  # rad/ns\n        return hz_to_radns(self.alpha_hz)\n    @property\n    def sigma_ns(self):\n        return self.sigma_frac * self.t_gate_ns\n\nCFG = Config()\n'


In [ ]:
# Supplied source file: drift.py
CORE_SOURCE['drift.py'] = '"""Seeded drift generators: OU (exact), random telegraph, gain."""\nfrom dataclasses import dataclass\nimport numpy as np\nfrom scipy.signal import lfilter\nfrom config import CFG, Config\n\ndef ou_exact(n, dt, tau, sigma, rng, mu=0.0):\n    """Exact OU sampling x_{k+1}=mu+(x_k-mu)e^{-dt/tau}+sigma*sqrt(1-e^{-2dt/tau})*xi.\n    x_0 is drawn from the stationary law N(mu, sigma^2). Returns n samples."""\n    a = np.exp(-dt / tau)\n    b = sigma * np.sqrt(1.0 - a * a)\n    x0 = rng.normal(0.0, sigma)\n    xi = rng.standard_normal(n - 1)\n    y, _ = lfilter([b], [1.0, -a], xi, zi=[a * x0])   # y[k] = a*y[k-1] + b*xi[k]\n    return mu + np.concatenate(([x0], y))\n\ndef telegraph(t, mean_dwell, rng, amp=1.0):\n    """Two-state {0, amp} telegraph; exponential dwell times, stationary 50/50 start.\n    (For a memoryless process the residual dwell at t=0 is also exponential.)"""\n    s0 = int(rng.integers(0, 2))\n    n_exp = int(2 * t[-1] / mean_dwell) + 50\n    switch = np.cumsum(rng.exponential(mean_dwell, n_exp))\n    assert switch[-1] > t[-1]\n    k = np.searchsorted(switch, t, side="right")\n    return amp * ((s0 + k) % 2).astype(float), switch[switch <= t[-1]]\n\n@dataclass\nclass DriftTrace:\n    t_s: np.ndarray        # grid [s]\n    delta_hz: np.ndarray   # Delta/2pi = xf + J  [Hz]\n    gain: np.ndarray       # g(t)\n    xf_hz: np.ndarray\n    J_hz: np.ndarray\n    xg: np.ndarray\n    phi: float\n    switch_times_s: np.ndarray\n    seed: int\n\n    def delta_hz_at(self, t_s):\n        return np.interp(t_s, self.t_s, self.delta_hz)\n    def gain_at(self, t_s):\n        return np.interp(t_s, self.t_s, self.gain)\n\ndef make_trace(seed=CFG.official_seed, cfg: Config = CFG, duration_s=None) -> DriftTrace:\n    """Independent child streams per process => changing one process never shifts another."""\n    dur = (cfg.day_s + cfg.drift_margin_s) if duration_s is None else duration_s\n    n = int(round(dur / cfg.drift_dt_s)) + 1\n    t = np.arange(n) * cfg.drift_dt_s\n    ss = np.random.SeedSequence(seed)\n    r_xf, r_j, r_xg, r_phi = [np.random.default_rng(s) for s in ss.spawn(4)]\n    xf = ou_exact(n, cfg.drift_dt_s, cfg.xf_tau_s, cfg.xf_sigma_hz, r_xf)\n    J, sw = telegraph(t, cfg.rtn_dwell_s, r_j, cfg.rtn_amp_hz)\n    xg = ou_exact(n, cfg.drift_dt_s, cfg.xg_tau_s, cfg.xg_sigma, r_xg)\n    phi = float(r_phi.uniform(0, 2 * np.pi))\n    g = 1.0 + cfg.gain_sin_amp * np.sin(2 * np.pi * t / cfg.day_s + phi) + xg\n    return DriftTrace(t, xf + J, g, xf, J, xg, phi, sw, seed)\n'


In [ ]:
# Supplied source file: transmon.py
CORE_SOURCE['transmon.py'] = '"""Three-level operators, Hamiltonian (Eq. 1) and Gaussian/DRAG pulses."""\nimport numpy as np\nfrom scipy.special import erf\nfrom config import CFG, Config\n\nD = 3\na = np.diag(np.sqrt(np.arange(1, D)), k=1).astype(complex)   # a|n> = sqrt(n)|n-1>\nad = a.conj().T\nN_OP = ad @ a                       # diag(0,1,2)\nANH = ad @ ad @ a @ a               # diag(0,0,2)\nXQ = a + ad                         # drive-x operator\nYQ = 1j * (ad - a)                  # drive-y operator\nP = np.diag([1, 1, 0]).astype(complex)   # projector on qubit subspace\n\ndef hamiltonian(delta, g, ox, oy, alpha):\n    """H(t) of Eq. (1), broadcast over any leading shape of delta,g,ox,oy [rad/ns].\n    Returns array (..., 3, 3)."""\n    delta, g, ox, oy = (np.asarray(v, float)[..., None, None] for v in (delta, g, ox, oy))\n    return delta * N_OP + (alpha / 2.0) * ANH + (g / 2.0) * (ox * XQ + oy * YQ)\n\ndef gaussian_pulse(theta, cfg: Config = CFG, n_steps=None, beta=0.0):\n    """Midpoint-sampled Gaussian X-quadrature and DRAG Y-quadrature [rad/ns].\n    Omega_x = A exp(-(t-t0)^2/2s^2) on [0,tg], area over the truncated window = theta.\n    Omega_y = -beta * dOmega_x/dt  (beta in ns). Returns (ox, oy, dt)."""\n    n = cfg.n_steps if n_steps is None else n_steps\n    tg, s = cfg.t_gate_ns, cfg.sigma_ns\n    dt = tg / n\n    t = (np.arange(n) + 0.5) * dt\n    t0 = tg / 2\n    A = theta / (s * np.sqrt(2 * np.pi) * erf(tg / (2 * np.sqrt(2) * s)))\n    ox = A * np.exp(-(t - t0) ** 2 / (2 * s * s))\n    dox = -(t - t0) / s ** 2 * ox          # analytic derivative\n    oy = -beta * dox\n    return ox, oy, dt\n'


In [ ]:
# Supplied source file: propagator.py
CORE_SOURCE['propagator.py'] = '"""Batched piecewise-constant propagator U = U_N ... U_1, U_k = exp(-i H_k dt)."""\nimport numpy as np\nfrom config import CFG\nfrom transmon import hamiltonian\n\ndef expm_herm(H, dt):\n    """exp(-i H dt) for Hermitian H of shape (...,3,3) via eigh (exactly unitary)."""\n    w, V = np.linalg.eigh(H)\n    ph = np.exp(-1j * w * dt)\n    return (V * ph[..., None, :]) @ V.conj().swapaxes(-1, -2)\n\ndef propagate(delta, g, ox, oy, dt, alpha=None):\n    """Full-gate propagator. delta, g: scalars or (B,) arrays; ox, oy: (N,) pulses.\n    Returns U of shape (B,3,3) (or (3,3) for scalar inputs). Later steps act on the left."""\n    alpha = CFG.alpha if alpha is None else alpha\n    scalar = np.ndim(delta) == 0 and np.ndim(g) == 0\n    d = np.atleast_1d(np.asarray(delta, float))\n    gg = np.atleast_1d(np.asarray(g, float))\n    d, gg = np.broadcast_arrays(d, gg)\n    H = hamiltonian(d[:, None], gg[:, None], ox[None, :], oy[None, :], alpha)  # (B,N,3,3)\n    Uk = expm_herm(H, dt)\n    U = Uk[:, 0]\n    for k in range(1, Uk.shape[1]):\n        U = Uk[:, k] @ U\n    return U[0] if scalar else U\n\ndef propagate_states(psi0, delta, g, ox, oy, dt, alpha=None):\n    """State trajectory (N+1,3) for a scalar-parameter gate; used for P0,P1,P2(t) plots."""\n    alpha = CFG.alpha if alpha is None else alpha\n    H = hamiltonian(delta, g, ox, oy, alpha)\n    Uk = expm_herm(H, dt)\n    psi = [np.asarray(psi0, complex)]\n    for k in range(len(ox)):\n        psi.append(Uk[k] @ psi[-1])\n    return np.array(psi)\n\ndef idle_propagator(delta, tau_ns, alpha=None):\n    """Free evolution (no drive); diagonal, exact."""\n    alpha = CFG.alpha if alpha is None else alpha\n    d = np.asarray(delta, float)[..., None]\n    E = np.stack([0 * d[..., 0], d[..., 0], 2 * d[..., 0] + alpha], axis=-1)\n    return np.exp(-1j * E * tau_ns)[..., None, :] * np.eye(3)\n'


In [ ]:
# Execute in dependency order as real in-memory modules so the original imports work.
import types
config_mod = types.ModuleType("config")
exec(CORE_SOURCE["config.py"], config_mod.__dict__)
sys.modules["config"] = config_mod
CFG = config_mod.CFG
hz_to_radns = config_mod.hz_to_radns
radns_to_hz = config_mod.radns_to_hz

drift_mod = types.ModuleType("drift")
exec(CORE_SOURCE["drift.py"], drift_mod.__dict__)
sys.modules["drift"] = drift_mod

T = types.ModuleType("transmon")
exec(CORE_SOURCE["transmon.py"], T.__dict__)
sys.modules["transmon"] = T

propagator_mod = types.ModuleType("propagator")
exec(CORE_SOURCE["propagator.py"], propagator_mod.__dict__)
sys.modules["propagator"] = propagator_mod

# Expose core APIs used by later notebook cells.
make_trace = drift_mod.make_trace
ou_exact = drift_mod.ou_exact
telegraph = drift_mod.telegraph
propagate = propagator_mod.propagate
expm_herm = propagator_mod.expm_herm
idle_propagator = propagator_mod.idle_propagator
print("Embedded core modules loaded.")
print(f"alpha/2π = {CFG.alpha_hz/1e6:.1f} MHz; gate = {CFG.t_gate_ns:g} ns; steps = {CFG.n_steps}")


## 3. Model and drift trace — official seed 2026

The frequency detuning combines an Ornstein–Uhlenbeck (OU) wander and a two-state telegraph component. Drive gain combines a 24-hour sinusoid and OU noise. The official seed is 2026, as specified in the report.


In [ ]:
trace = make_trace(CFG.official_seed)
hours = trace.t_s / 3600.0
mask = hours <= 24.0
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(hours[mask], trace.xf_hz[mask] / 1e3, lw=.8, label="OU frequency wander")
axes[0].plot(hours[mask], trace.J_hz[mask] / 1e3, lw=1.2, label="telegraph component")
axes[0].plot(hours[mask], trace.delta_hz[mask] / 1e3, lw=.9, label="total detuning")
axes[0].set_ylabel("Frequency / 2π (kHz)")
axes[0].legend()
axes[1].plot(hours[mask], trace.gain[mask], label="drive gain g(t)")
axes[1].set_xlabel("Time (hours)")
axes[1].set_ylabel("Drive gain")
axes[1].legend()
fig.suptitle("Official drift trace (seed 2026)")
fig.tight_layout()
plt.show()


## 4. Three-level Hamiltonian, Gaussian pulse, and DRAG

The supplied model uses a truncated three-level transmon with anharmonicity α/2π = −300 MHz and a 20 ns Gaussian pulse whose standard deviation is 5 ns. The DRAG quadrature is Ωᵧ = −β dΩₓ/dt. The report notes that the numerical optimum is β ≈ −0.313 ns, rather than the first-order estimate of approximately −0.53 ns. fileciteturn6file0L44-L50


In [ ]:
for theta in (np.pi/2, np.pi):
    ox, oy, dt = T.gaussian_pulse(theta)
    print(f"theta={theta:.6f} rad | integrated X area={ox.sum()*dt:.8f} rad | dt={dt:.3f} ns")
ox, oy, dt = T.gaussian_pulse(np.pi, beta=-0.313)
plt.figure(figsize=(9, 3.5))
t_ns = (np.arange(len(ox)) + .5) * dt
plt.plot(t_ns, ox, label="Ωx")
plt.plot(t_ns, oy, label="Ωy (DRAG, β=-0.313 ns)")
plt.xlabel("Time (ns)")
plt.ylabel("Drive amplitude (rad/ns)")
plt.title("20 ns truncated Gaussian π pulse")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Propagator validation and static gate diagnostic

The propagator forms a product of unitary 3×3 exponentials. This cell checks unitarity over a batch of detunings and gains and plots leakage from the computational subspace for a static π pulse. The displayed leakage proxy is a diagnostic computed from the supplied propagator; it is not a replacement for the report's hidden-oracle gate-error metric.


In [ ]:
ox, oy, dt = T.gaussian_pulse(np.pi, beta=-0.313)
detunings = hz_to_radns(np.linspace(-1e6, 1e6, 21))
gains = np.linspace(.98, 1.02, 21)
U = propagate(detunings, gains, ox, oy, dt)
unitarity_residual = np.max(np.abs(U @ U.conj().swapaxes(-1, -2) - np.eye(3)))
print("Batched propagator shape:", U.shape)
print("Maximum unitarity residual:", f"{unitarity_residual:.3e}")
U0 = propagate(0.0, 1.0, ox, oy, dt)
psi = U0[:, 0]
print("Static π pulse: |0> survival =", abs(psi[0])**2)
print("Static π pulse: |1> population =", abs(psi[1])**2)
print("Static π pulse: |2> leakage =", abs(psi[2])**2)
assert unitarity_residual < 1e-10


## 6. OU process sanity check

The OU generator uses exact discrete-time transitions and seeded random streams. The following diagnostic compares empirical autocorrelation with the theoretical exponential decay for the supplied frequency-drift time constant.


In [ ]:
rng = np.random.default_rng(7)
x = ou_exact(200_000, 1.0, CFG.xf_tau_s, CFG.xf_sigma_hz, rng)
lags = np.array([0, 300, 900, 1800, 3600, 7200, 10800, 14400])
empirical = []
for lag in lags:
    empirical.append(1.0 if lag == 0 else np.corrcoef(x[:-lag], x[lag:])[0, 1])
theory = np.exp(-lags / CFG.xf_tau_s)
plt.figure(figsize=(7, 3.5))
plt.plot(lags/3600, empirical, "o-", label="empirical")
plt.plot(lags/3600, theory, "--", label="theory")
plt.xlabel("Lag (hours)")
plt.ylabel("Autocorrelation")
plt.title("OU autocorrelation sanity check")
plt.legend()
plt.tight_layout()
plt.show()
print("Empirical SD:", f"{np.std(x, ddof=1)/1e3:.2f} kHz; target 150 kHz")


## 7. Core model checks

These assertions cover key invariants of the embedded source: operator definitions, pulse area, Hermiticity, unitarity, and deterministic drift generation. They are lightweight notebook checks; the complete supplied test file can be run separately using pytest from the project directory.


In [ ]:
assert np.allclose(T.N_OP, np.diag([0, 1, 2]))
assert np.allclose(T.ANH, np.diag([0, 0, 2]))
assert np.allclose(T.XQ[:2, :2], [[0, 1], [1, 0]])
assert np.allclose(T.YQ[:2, :2], [[0, -1j], [1j, 0]])
for theta in (np.pi/2, np.pi, .3):
    pulse_x, pulse_y, pulse_dt = T.gaussian_pulse(theta)
    assert np.isclose(pulse_x.sum()*pulse_dt, theta, rtol=1e-4)
    assert np.allclose(pulse_y, 0)
H = T.hamiltonian(hz_to_radns(1234e3), 1.0, 0.0, 0.0, CFG.alpha)
assert np.allclose(H, H.conj().T)
assert np.allclose(U @ U.conj().swapaxes(-1, -2), np.eye(3), atol=1e-10)
trace_again = make_trace(CFG.official_seed)
assert np.array_equal(trace.delta_hz, trace_again.delta_hz)
print("PASS: notebook core checks")


## 8. Calibration primitives and adaptive Ramsey — reported findings

The report describes three calibration primitives: amplitude calibration by error amplification, Bayesian Ramsey frequency estimation, and DRAG calibration. It reports amplitude RMS error improving from about 9×10⁻³ to 1.4×10⁻³ as the shot budget rises from 60 to 1,920 shots. At 0.5 kHz RMS frequency error, adaptive Bayesian Ramsey used about 420 shots versus about 1,110 for a fixed logarithmic grid (roughly 2.7× fewer); at very small budgets, adaptive alias failures mean a fixed grid can be safer. These are **transcribed report results**, not recalculated by the available core source. fileciteturn6file1L47-L64


## 9. Policy comparison — reported aggregate table

The provided report aggregates 22 drift traces (seed 2026 plus seeds 1–21) and 22 policy configurations, totalling 484 simulated days. The table below transcribes selected periodic policies. P0 never recalibrates; P1 periodically recalibrates. Values are report results, not outputs of a rerun in this notebook. fileciteturn6file0L6-L22


In [ ]:
import pandas as pd
policy_rows = [
    {"Policy":"P0 (never)","Calibration time (% day)":0.0015,"Mean gate error":8.84e-4,"Time below 1e-3 (%)":65.8,"Recalibrations/day":0,"QAOA shortfall (%)":0.227},
    {"Policy":"P1 (10 min)","Calibration time (% day)":0.0441,"Mean gate error":1.23e-4,"Time below 1e-3 (%)":99.5,"Recalibrations/day":143,"QAOA shortfall (%)":0.016},
    {"Policy":"P1 (30 min)","Calibration time (% day)":0.0153,"Mean gate error":1.69e-4,"Time below 1e-3 (%)":98.0,"Recalibrations/day":47,"QAOA shortfall (%)":0.038},
    {"Policy":"P1 (60 min)","Calibration time (% day)":0.0086,"Mean gate error":2.15e-4,"Time below 1e-3 (%)":96.0,"Recalibrations/day":23,"QAOA shortfall (%)":0.059},
    {"Policy":"P1 (120 min)","Calibration time (% day)":0.0048,"Mean gate error":3.12e-4,"Time below 1e-3 (%)":92.3,"Recalibrations/day":11,"QAOA shortfall (%)":0.100},
]
policy_df = pd.DataFrame(policy_rows)
display(policy_df)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(policy_df["Calibration time (% day)"], policy_df["Mean gate error"], "o-")
for _, row in policy_df.iterrows():
    ax.annotate(row["Policy"], (row["Calibration time (% day)"], row["Mean gate error"]), xytext=(4,4), textcoords="offset points", fontsize=8)
ax.set_xlabel("Calibration time (% of day)")
ax.set_ylabel("Mean gate error")
ax.set_title("Reported calibration-time / gate-error trade-off")
ax.grid(True, alpha=.3)
plt.tight_layout()
plt.show()


## 10. QAOA impact and D6 memo — reported findings

The reported p=1 QAOA instance is MaxCut on a four-qubit ring. The report states that hourly recalibration reduces mean gate error to 2.15×10⁻⁴ versus 8.84×10⁻⁴ with no recalibration. Yet the QAOA is comparatively forgiving: the no-recalibration policy has mean ideal-cost shortfall of only 0.23%. The isolated telegraph frequency jump is reported to hurt algorithm cost most (0.176% of ideal on average), while gain drift dominates raw gate error. These are simulation results reported in the supplied materials. fileciteturn6file10L1-L9


## 11. Figures supplied with the report

These figures were supplied as rendered artifacts. They are included as visual evidence, not regenerated by this notebook. If the notebook is run outside the original bundle directory, copy the image files into the working directory or update `FIGURE_DIR`.


In [ ]:
from IPython.display import display, Image
FIGURE_DIR = Path.cwd() / "figures"
if not FIGURE_DIR.exists():
    FIGURE_DIR = Path.cwd().parent / "figures"
figure_files = [
    "fig3_static_pi_pulse.png", "fig3b_P2_vs_beta.png",
    "fig4_calibration_convergence.png", "fig5_ramsey_alias.png",
    "fig7_drag_scan.png", "fig8_adaptive_vs_fixed.png",
    "fig9_posterior_evolution.png", "fig10_policy_traces_official_seed.png",
    "fig11_pareto.png", "fig12_qaoa_vs_time.png",
]
for fname in figure_files:
    path = FIGURE_DIR / fname
    if path.exists():
        print(fname)
        display(Image(filename=str(path), width=700))
    else:
        print(f"Not embedded/displayed (file not in current directory): {fname}")


## 12. Scope, limitations, and exact full reproduction path

**What this notebook actually reruns:** the supplied configuration, OU/telegraph/gain drift, three-level Hamiltonian, Gaussian/DRAG pulse, propagator, static pulse diagnostics, and notebook core assertions.

**What remains report-derived:** the full shot-level device interface, original calibration agent and Bayesian design implementation, P0/P1/P2 closed-loop policy sweep, hidden-oracle gate-error evaluation, full QAOA evaluation, and ablation studies. The supplied README lists modules such as `oracle.py`, `plant.py`, `agent.py`, `policies.py`, `closed_loop.py`, `qaoa.py`, and `run_sweep.py`, but those original files were not present in the uploaded bundle. The notebook therefore does not claim to reproduce those missing stages. fileciteturn6file9L4-L16

To reproduce the original full sweep when the missing original implementation modules are available, run the project's documented command from the project root:
`python run_sweep.py results/sweep.jsonl 2026,1,...,21`

The reported results are useful for analysis, but the missing implementation files are required for an independent end-to-end rerun.
